# Comparação das soluções para a aorta

Compara quatro estratégias nas mesmas imagens de treino e validação:

- pipeline normal;
- filtro agressivo de círculos;
- filtro com cobertura mínima de 80%;
- filtro com cobertura mínima de 65% e fallback;
- controlador adaptativo do level set, sem filtro de círculos ou correção posterior.

A inspeção manual é usada somente para classificar a qualidade da máscara da aorta. O sucesso dos óstios é calculado diretamente a partir de `ostia_detection_status` nos CSVs de cada run.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Localiza a raiz antes de importar os módulos do projeto.
current = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in [current, *current.parents]
    if (path / "src").exists() and (path / "output").exists()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    get_aorta_visual_review,
    load_aorta_visual_reviews,
    resolve_aorta_review_summary_path,
)
from utils.project.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
pd.set_option("display.max_columns", 40)

## 1. Configuração e carregamento

Os caminhos dos runs e as classificações visuais ficam centralizados em `config/aorta_visual_reviews.json`. Cada CSV deve conter exatamente os exames classificados para sua variante e subconjunto.

In [ ]:
REVIEW_CONFIG_PATH = REPO_ROOT / "config/aorta_visual_reviews.json"
SPLITS = ("train", "val")
VARIANTS = (
    "normal",
    "aggressive",
    "coverage_080",
    "coverage_065_fallback",
    "adaptive_current",
)

SPLIT_NAMES = {"train": "Treino", "val": "Validação"}
DISPLAY_NAMES = {
    "normal": "Normal",
    "aggressive": "Filtro agressivo",
    "coverage_080": "Cobertura 80%",
    "coverage_065_fallback": "Cobertura 65% + fallback",
    "adaptive_current": "Level set adaptativo",
}
SUCCESS_STATUSES = {
    "both correct",
    "both tolerable",
    "both ostia correct",
    "both ostia tolerable",
}
SUMMARY_COLUMNS = [
    "IMG_ID",
    "artery_dice",
    "ostia_detection_status",
    "aorta_mask_voxel_count",
    "aorta_volume_fraction",
]

review_catalog = load_aorta_visual_reviews(REVIEW_CONFIG_PATH)
reviews = {
    (split, variant): get_aorta_visual_review(review_catalog, variant, split)
    for split in SPLITS
    for variant in VARIANTS
}


def load_solution(split, variant):
    """Carrega métricas automáticas e acrescenta apenas o rótulo visual da aorta."""
    review = reviews[(split, variant)]
    summary_path = resolve_aorta_review_summary_path(REPO_ROOT, review, split)
    frame = pd.read_csv(summary_path)

    missing_columns = set(SUMMARY_COLUMNS).difference(frame.columns)
    if missing_columns:
        raise ValueError(
            f"Colunas ausentes em {variant}/{split}: {sorted(missing_columns)}"
        )

    # Limita a análise às métricas comuns aos quatro experimentos.
    frame = frame[SUMMARY_COLUMNS].copy()
    frame["IMG_ID"] = pd.to_numeric(frame["IMG_ID"], errors="raise").astype(int)

    expected_ids = review["aorta_good_ids"] | review["aorta_bad_ids"]
    observed_ids = set(frame["IMG_ID"])
    if observed_ids != expected_ids:
        raise ValueError(
            f"IDs incompatíveis em {variant}/{split}: "
            f"ausentes={sorted(expected_ids - observed_ids)}; "
            f"não revisados={sorted(observed_ids - expected_ids)}"
        )

    # Both correct e both tolerable representam sucesso dos dois óstios.
    normalized_status = (
        frame["ostia_detection_status"]
        .astype(str)
        .str.lower()
        .str.replace("_", " ", regex=False)
        .str.strip()
    )
    frame["ostia_success"] = normalized_status.isin(SUCCESS_STATUSES)
    frame["aorta_visual_good"] = frame["IMG_ID"].isin(review["aorta_good_ids"])
    frame["variant"] = variant
    frame["split"] = split
    return frame


solutions = {
    (split, variant): load_solution(split, variant)
    for split in SPLITS
    for variant in VARIANTS
}

for split in SPLITS:
    cohort_ids = [set(solutions[(split, variant)]["IMG_ID"]) for variant in VARIANTS]
    if any(ids != cohort_ids[0] for ids in cohort_ids[1:]):
        raise ValueError(f"As variantes de {split} não usam a mesma coorte.")
    print(f"{SPLIT_NAMES[split]}: {len(cohort_ids[0])} exames em cada variante")

## 2. Resultados gerais

A tabela reúne os três resultados principais: proporção de aortas consideradas boas na inspeção visual, sucesso automático dos óstios e Dice da segmentação arterial.

In [ ]:
overview_rows = []
for split in SPLITS:
    for variant in VARIANTS:
        frame = solutions[(split, variant)]
        overview_rows.append(
            {
                "subconjunto": SPLIT_NAMES[split],
                "solução": DISPLAY_NAMES[variant],
                "imagens": len(frame),
                "aortas_boas": int(frame["aorta_visual_good"].sum()),
                "aortas_boas_%": 100 * frame["aorta_visual_good"].mean(),
                "sucesso_óstios_csv": int(frame["ostia_success"].sum()),
                "sucesso_óstios_%": 100 * frame["ostia_success"].mean(),
                "dice_médio": frame["artery_dice"].mean(),
                "dice_mediano": frame["artery_dice"].median(),
                "volume_aorta_médio_%": 100 * frame["aorta_volume_fraction"].mean(),
            }
        )

overview_df = pd.DataFrame(overview_rows)
display(overview_df.round(4))

## 3. Diferença em relação ao pipeline normal

Os deltas são calculados de forma pareada pelos mesmos `IMG_IDs`. Valores positivos em Dice e sucesso dos óstios favorecem a variante. As listas de aortas corrigidas e pioradas vêm exclusivamente da inspeção visual.

In [ ]:
comparison_rows = []
for split in SPLITS:
    baseline = solutions[(split, "normal")].set_index("IMG_ID").sort_index()
    baseline_review = reviews[(split, "normal")]

    for variant in VARIANTS[1:]:
        candidate = solutions[(split, variant)].set_index("IMG_ID").sort_index()
        candidate_review = reviews[(split, variant)]
        paired_ids = baseline.index.intersection(candidate.index)

        normal_bad = baseline_review["aorta_bad_ids"]
        candidate_bad = candidate_review["aorta_bad_ids"]
        comparison_rows.append(
            {
                "subconjunto": SPLIT_NAMES[split],
                "solução": DISPLAY_NAMES[variant],
                "delta_dice_médio": (
                    candidate.loc[paired_ids, "artery_dice"]
                    - baseline.loc[paired_ids, "artery_dice"]
                ).mean(),
                "delta_sucesso_óstios_pp": 100 * (
                    candidate.loc[paired_ids, "ostia_success"].mean()
                    - baseline.loc[paired_ids, "ostia_success"].mean()
                ),
                "delta_aortas_boas": (
                    int(candidate.loc[paired_ids, "aorta_visual_good"].sum())
                    - int(baseline.loc[paired_ids, "aorta_visual_good"].sum())
                ),
                "máscaras_com_voxels_diferentes": int(
                    (
                        candidate.loc[paired_ids, "aorta_mask_voxel_count"]
                        != baseline.loc[paired_ids, "aorta_mask_voxel_count"]
                    ).sum()
                ),
                "aortas_corrigidas": sorted(normal_bad - candidate_bad),
                "aortas_pioradas": sorted(candidate_bad - normal_bad),
            }
        )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(4))

## 4. Comparação visual das métricas

Cada linha representa um subconjunto. As escalas percentuais da qualidade da aorta e dos óstios são separadas do Dice para evitar interpretações equivocadas.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8), constrained_layout=True)
colors = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756"]
metrics = (
    ("aortas_boas_%", "Aortas visualmente boas (%)", (0, 100)),
    ("sucesso_óstios_%", "Sucesso dos óstios pelo CSV (%)", (0, 100)),
    ("dice_médio", "Dice médio", (0, 1)),
)

for row, split in enumerate(SPLITS):
    split_df = overview_df[overview_df["subconjunto"].eq(SPLIT_NAMES[split])]
    for column, (metric, title, limits) in enumerate(metrics):
        axis = axes[row, column]
        bars = axis.bar(split_df["solução"], split_df[metric], color=colors)
        axis.set_title(f"{SPLIT_NAMES[split]} — {title}", fontsize=11)
        axis.set_ylim(*limits)
        axis.tick_params(axis="x", rotation=24, labelsize=9)
        axis.grid(axis="y", alpha=0.25)

        decimals = 3 if metric == "dice_médio" else 1
        labels = [f"{value:.{decimals}f}" for value in split_df[metric]]
        axis.bar_label(bars, labels=labels, padding=3, fontsize=9)

plt.show()

## 5. Síntese

A seleção de uma estratégia não deve considerar apenas a aparência da aorta. Esta célula identifica, em cada subconjunto, qual variante obteve o maior Dice, a maior taxa visual de aortas boas e a maior taxa automática de sucesso dos óstios.

In [ ]:
for split in SPLITS:
    split_df = overview_df[overview_df["subconjunto"].eq(SPLIT_NAMES[split])]
    best_dice = split_df.loc[split_df["dice_médio"].idxmax()]
    best_aorta = split_df.loc[split_df["aortas_boas_%"].idxmax()]
    best_ostia = split_df.loc[split_df["sucesso_óstios_%"].idxmax()]

    print(SPLIT_NAMES[split])
    print(f"- Maior Dice: {best_dice['solução']} ({best_dice['dice_médio']:.4f})")
    print(
        f"- Mais aortas visualmente boas: {best_aorta['solução']} "
        f"({best_aorta['aortas_boas_%']:.1f}%)"
    )
    print(
        f"- Maior sucesso dos óstios: {best_ostia['solução']} "
        f"({best_ostia['sucesso_óstios_%']:.1f}%)"
    )